# 最小的Agent工作台

最小的agent工作台包含三个文件： 一个根指令路由器、一个状态文件以及一个任务看板。其他所有的东西都是堆在上面的。如果一个仓库不能同时拥有这三个东西，那么没有模型能够拯救它。

## 问题描述

大多数团队摸到工作台的方式是写一个3000行的`AGENTS.md` 文件，然后就认为完成了。模型加载它，然后忽略掉模型不能总结的部分，结构就是在它总是失败的地方再次失败。

你需要的恰恰相反。一个小的根路由文件，在相关的时候将agent交接给更深层的文件。在agent读之间或者写完以后把状态持久化。以及一个任务看板告诉你哪些正在执行、哪些被阻塞了、哪些是接下来要做的。

## 基本概念

```mermaid
flowchart LR
A[Agent Loop]-->B[AGENTS.md]-->C[agent_state.json]
B-->D[task_board.json]
C-->A
D-->A
```

### AGENTS.MD 是一个路由器而不是操作手册

好的`AGENTS.MD`应该简短，应该指出：

- 状态文件（你现在在哪）
- 任务看板（还剩哪些）
- 更深层的规则
- 验证命令（怎么知道正确工作了）

任何其他的东西应该放在更深的文档中，只在需要的时候加载上来。如果使用长篇的指引，会因为注意力稀释导致忽略，而更短的路由往往能够被遵守。

### AGENT_STATE.JSON 是记录系统

状态需要关注的内容：当前任务id，摸到的文件，做出的假设，阻拦项以及下一个行动。agent每轮都会读取它。下一个会话靠直接读它而不是重播聊天。

状态选择用一个文件存储的原因是聊天历史不可靠。当会话结束的时候，会话历史会被裁剪，但是状态文件不会。

### TASK_BOARD.JSON 是序列

任务看板包含每个标记为`todo|in_progress|done|blocked`的任务。这就是当状态为空时，agent获取任务的队列，以及当你想知道agent是否仍在正常工作时的需要读取的队列。

看板上的任务需要有：目标、所有者（`builder/reviewer/human`）、以及接受标准。看板故意设计得很小，当它增长超过屏幕的时候，表明你遇到了规划问题，而不是看板有问题。

### 三个文件是地板不是天花板

后续会陆续加入范围契约、反馈运行、验证门控、校验检查列表、交接包裹。这三个文件是后续内容假设需要的。



# 开始编码

对应本章核心：**三文件地板（AGENTS.md 路由器 / agent_state.json / task_board.json）**、**短路由优于长手册**、**状态跨会话持久（不重播聊天）**、**看板认领 → 验收 → done/blocked**。  
先用玩具在临时目录跑通读写与会话交接；再用 **LangChain 工具 + DeepSeek** 让 agent 只通过工作台文件推进任务。不硬凑 PyTorch。


## 1. 教学玩具：最小工作台三文件

- **AGENTS.md**：只指路（state / board / deep rules / verify），不当百科。
- **agent_state.json**：current_task、touched、assumptions、blockers、next_action。
- **task_board.json**：`todo|in_progress|done|blocked` + owner + acceptance。


In [1]:
from __future__ import annotations

import json
import shutil
import tempfile
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Literal

Status = Literal["todo", "in_progress", "done", "blocked"]
Owner = Literal["builder", "reviewer", "human"]


@dataclass
class Task:
    """看板上的一条任务。"""

    id: str
    goal: str
    owner: Owner = "builder"
    status: Status = "todo"
    acceptance: list[str] = field(default_factory=list)


@dataclass
class AgentState:
    """记录系统：跨会话可读。"""

    current_task_id: str | None = None
    touched_files: list[str] = field(default_factory=list)
    assumptions: list[str] = field(default_factory=list)
    blockers: list[str] = field(default_factory=list)
    next_action: str = ""
    last_session: str = ""


DEFAULT_AGENTS_MD = """# AGENTS.md (router)

- State: `agent_state.json` — where you are now
- Board: `task_board.json` — what remains
- Deep rules: `docs/rules.md` — load only when needed
- Verify: `python -m pytest -q` — how you know it worked

Do not treat this file as an operations manual. Follow links.
"""


class MinimalWorkbench:
    """三文件地板：根路由 + 状态 + 看板。"""

    def __init__(self, root: Path) -> None:
        self.root = Path(root)
        self.agents_md = self.root / "AGENTS.md"
        self.state_path = self.root / "agent_state.json"
        self.board_path = self.root / "task_board.json"

    def init(
        self,
        *,
        tasks: list[Task] | None = None,
        agents_md: str | None = None,
    ) -> None:
        """
        在 root 写入三文件地板。

        Args:
            tasks: 初始看板。
            agents_md: 可选自定义路由文本。
        """
        self.root.mkdir(parents=True, exist_ok=True)
        self.agents_md.write_text(agents_md or DEFAULT_AGENTS_MD, encoding="utf-8")
        self.write_state(AgentState())
        board = {"tasks": [asdict(t) for t in (tasks or [])]}
        self.board_path.write_text(json.dumps(board, ensure_ascii=False, indent=2), encoding="utf-8")

    def read_router(self) -> dict[str, str]:
        """
        解析 AGENTS.md 里的关键指针（演示「路由器」语义）。

        Returns:
            links: state/board/deep/verify 路径或命令。
        """
        text = self.agents_md.read_text(encoding="utf-8")
        links = {"state": "", "board": "", "deep": "", "verify": ""}
        for line in text.splitlines():
            low = line.lower()
            if "state:" in low and "`" in line:
                links["state"] = line.split("`")[1]
            elif "board:" in low and "`" in line:
                links["board"] = line.split("`")[1]
            elif "deep" in low and "`" in line:
                links["deep"] = line.split("`")[1]
            elif "verify:" in low and "`" in line:
                links["verify"] = line.split("`")[1]
        return links

    def is_short_router(self, *, max_lines: int = 40, max_chars: int = 2000) -> bool:
        """短路由启发式：过长则像操作手册，易被忽略。"""
        text = self.agents_md.read_text(encoding="utf-8")
        return len(text.splitlines()) <= max_lines and len(text) <= max_chars

    def read_state(self) -> AgentState:
        data = json.loads(self.state_path.read_text(encoding="utf-8"))
        return AgentState(**data)

    def write_state(self, state: AgentState) -> None:
        self.state_path.write_text(
            json.dumps(asdict(state), ensure_ascii=False, indent=2), encoding="utf-8"
        )

    def read_board(self) -> list[Task]:
        data = json.loads(self.board_path.read_text(encoding="utf-8"))
        return [Task(**t) for t in data.get("tasks") or []]

    def write_board(self, tasks: list[Task]) -> None:
        self.board_path.write_text(
            json.dumps({"tasks": [asdict(t) for t in tasks]}, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

    def claim_next(self, *, session: str, owner: Owner = "builder") -> Task | None:
        """
        从看板认领下一条 todo；写入状态。状态空时靠看板取活。

        Returns:
            task: 认领到的任务；无则 None。
        """
        tasks = self.read_board()
        # 已有 in_progress 则续上
        for t in tasks:
            if t.status == "in_progress" and t.owner == owner:
                st = self.read_state()
                st.current_task_id = t.id
                st.last_session = session
                st.next_action = f"continue {t.id}"
                self.write_state(st)
                return t
        for t in tasks:
            if t.status == "todo" and t.owner == owner:
                t.status = "in_progress"
                self.write_board(tasks)
                st = AgentState(
                    current_task_id=t.id,
                    touched_files=[],
                    assumptions=[],
                    blockers=[],
                    next_action=f"work on {t.id}: {t.goal}",
                    last_session=session,
                )
                self.write_state(st)
                return t
        return None

    def touch(self, path: str) -> None:
        st = self.read_state()
        if path not in st.touched_files:
            st.touched_files.append(path)
        self.write_state(st)

    def note_assumption(self, text: str) -> None:
        st = self.read_state()
        st.assumptions.append(text)
        self.write_state(st)

    def block(self, reason: str) -> None:
        st = self.read_state()
        st.blockers.append(reason)
        st.next_action = "blocked"
        tid = st.current_task_id
        self.write_state(st)
        if tid:
            tasks = self.read_board()
            for t in tasks:
                if t.id == tid:
                    t.status = "blocked"
            self.write_board(tasks)

    def complete(self, *, evidence: list[str]) -> None:
        """
        验收：acceptance 每条都要在 evidence 里出现子串，才标 done。
        """
        st = self.read_state()
        tid = st.current_task_id
        if not tid:
            raise RuntimeError("no current task")
        tasks = self.read_board()
        task = next(t for t in tasks if t.id == tid)
        missing = [a for a in task.acceptance if not any(a in e for e in evidence)]
        if missing:
            raise RuntimeError(f"acceptance unmet: {missing}")
        task.status = "done"
        self.write_board(tasks)
        st.current_task_id = None
        st.next_action = "claim next"
        st.blockers = []
        self.write_state(st)

    def handoff_prompt(self) -> str:
        """
        新会话启动时读工作台，不重播聊天。

        Returns:
            prompt: 可注入系统侧的交接摘要。
        """
        links = self.read_router()
        st = self.read_state()
        tasks = self.read_board()
        open_tasks = [t for t in tasks if t.status in {"todo", "in_progress", "blocked"}]
        return (
            f"Router links: {links}\n"
            f"State: task={st.current_task_id} next={st.next_action} "
            f"touched={st.touched_files} blockers={st.blockers}\n"
            f"Open board: {[{'id': t.id, 'status': t.status, 'goal': t.goal} for t in open_tasks]}\n"
            "Resume from state+board, not from chat history."
        )


def make_demo_root() -> Path:
    """临时仓库根。"""
    return Path(tempfile.mkdtemp(prefix="min_wb_"))


print("minimal workbench ready | AGENTS.md + state + board")


minimal workbench ready | AGENTS.md + state + board


## 2. 玩具示例：短路由、认领、跨会话交接、验收门


In [ ]:
def demo_minimal_workbench() -> None:
    """断言三文件地板与跨会话恢复。"""
    root = make_demo_root()
    try:
        wb = MinimalWorkbench(root)
        wb.init(
            tasks=[
                Task(
                    id="T1",
                    goal="add input validation",
                    acceptance=["tests pass", "only src/api.py"],
                ),
                Task(id="T2", goal="write changelog", acceptance=["CHANGELOG"]),
            ]
        )
        assert {p.name for p in root.iterdir()} >= {"AGENTS.md", "agent_state.json", "task_board.json"}
        links = wb.read_router()
        assert links["state"] == "agent_state.json" and links["board"] == "task_board.json"
        assert wb.is_short_router()
        # 长手册启发式失败
        wb.agents_md.write_text("# manual\n" + ("x\n" * 80), encoding="utf-8")
        assert not wb.is_short_router()
        wb.agents_md.write_text(DEFAULT_AGENTS_MD, encoding="utf-8")
        print("short router ok")

        t = wb.claim_next(session="s1")
        assert t and t.id == "T1" and t.status == "in_progress"
        wb.touch("src/api.py")
        wb.note_assumption("validators live in src/api.py")
        st = wb.read_state()
        assert st.current_task_id == "T1" and "src/api.py" in st.touched_files
        print("claim + state write ok")

        # 新会话：不靠聊天，只读文件
        wb2 = MinimalWorkbench(root)
        prompt = wb2.handoff_prompt()
        assert "T1" in prompt and "src/api.py" in prompt and "chat history" in prompt
        t2 = wb2.claim_next(session="s2")  # 续上 in_progress
        assert t2 and t2.id == "T1"
        assert wb2.read_state().last_session == "s2"
        print("cross-session handoff ok")

        # 验收不过
        try:
            wb2.complete(evidence=["edited api"])
            raise AssertionError("should fail acceptance")
        except RuntimeError as e:
            assert "acceptance unmet" in str(e)
        wb2.complete(evidence=["tests pass on CI", "only src/api.py touched"])
        assert wb2.read_board()[0].status == "done"
        assert wb2.read_state().current_task_id is None
        print("acceptance gate ok")

        t3 = wb2.claim_next(session="s2")
        assert t3 and t3.id == "T2"
        print("TOY DEMO OK")
    finally:
        shutil.rmtree(root, ignore_errors=True)


demo_minimal_workbench()


## 3. 生产级：LangChain 工具读写工作台 + DeepSeek

工具只暴露工作台原语：`claim_task` / `read_handoff` / `touch_file` / `complete_task` / `board_status`。模型不能绕过验收门。需 `DEEPSEEK_API_KEY`。


In [3]:
import json
import os
import sys
from pathlib import Path
from typing import Any

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"
PROD_ROOT: Path | None = None
PROD_WB: MinimalWorkbench | None = None


def get_llm(*, temperature: float = 0.0) -> Any:
    """
    Returns:
        llm: DeepSeek chat model。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing; copy .env.example → .env")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


def reset_prod_workbench() -> str:
    """
    重建临时工作台，放入两条演示任务。

    Returns:
        json: root 路径。
    """
    global PROD_ROOT, PROD_WB
    if PROD_ROOT and PROD_ROOT.exists():
        shutil.rmtree(PROD_ROOT, ignore_errors=True)
    PROD_ROOT = make_demo_root()
    PROD_WB = MinimalWorkbench(PROD_ROOT)
    PROD_WB.init(
        tasks=[
            Task(
                id="VAL",
                goal="为登录接口补输入校验",
                owner="builder",
                acceptance=["pytest green", "src/auth.py"],
            ),
            Task(
                id="DOC",
                goal="更新 README 说明校验规则",
                owner="builder",
                acceptance=["README"],
            ),
        ]
    )
    return json.dumps({"root": str(PROD_ROOT)}, ensure_ascii=False)


def _wb() -> MinimalWorkbench:
    if PROD_WB is None:
        reset_prod_workbench()
    assert PROD_WB is not None
    return PROD_WB


class SessionArgs(BaseModel):
    session: str = "prod"


class TouchArgs(BaseModel):
    path: str


class CompleteArgs(BaseModel):
    evidence: list[str] = Field(description="必须覆盖 acceptance 的证据字符串列表")


class EmptyArgs(BaseModel):
    pass


def build_wb_tools() -> list[StructuredTool]:
    def _claim(**kwargs: Any) -> str:
        a = SessionArgs(**kwargs)
        task = _wb().claim_next(session=a.session)
        if task is None:
            return json.dumps({"ok": False, "error": "no todo"}, ensure_ascii=False)
        return json.dumps({"ok": True, "task": asdict(task), "handoff": _wb().handoff_prompt()}, ensure_ascii=False)

    def _handoff(**kwargs: Any) -> str:
        return _wb().handoff_prompt()

    def _touch(**kwargs: Any) -> str:
        p = TouchArgs(**kwargs).path
        _wb().touch(p)
        return json.dumps(asdict(_wb().read_state()), ensure_ascii=False)

    def _complete(**kwargs: Any) -> str:
        a = CompleteArgs(**kwargs)
        try:
            _wb().complete(evidence=a.evidence)
            return json.dumps({"ok": True, "state": asdict(_wb().read_state())}, ensure_ascii=False)
        except RuntimeError as e:
            return json.dumps({"ok": False, "error": str(e)}, ensure_ascii=False)

    def _board(**kwargs: Any) -> str:
        tasks = [asdict(t) for t in _wb().read_board()]
        return json.dumps({"tasks": tasks, "short_router": _wb().is_short_router()}, ensure_ascii=False)

    def _reset(**kwargs: Any) -> str:
        return reset_prod_workbench()

    return [
        StructuredTool.from_function(name="reset_workbench", description="Reset demo three-file workbench.", func=_reset, args_schema=EmptyArgs),
        StructuredTool.from_function(name="claim_task", description="Claim next todo into agent_state.json.", func=_claim, args_schema=SessionArgs),
        StructuredTool.from_function(name="read_handoff", description="Read state+board handoff (no chat replay).", func=_handoff, args_schema=EmptyArgs),
        StructuredTool.from_function(name="touch_file", description="Record a touched file into agent_state.json.", func=_touch, args_schema=TouchArgs),
        StructuredTool.from_function(name="complete_task", description="Mark current task done only if evidence meets acceptance.", func=_complete, args_schema=CompleteArgs),
        StructuredTool.from_function(name="board_status", description="Show task board and whether AGENTS.md is a short router.", func=_board, args_schema=EmptyArgs),
    ]


WB_TOOLS = build_wb_tools()


def build_control_agent() -> Any:
    system = (
        "You operate a minimal agent workbench (AGENTS.md + agent_state.json + task_board.json).\n"
        "Flow: reset_workbench -> claim_task -> touch_file -> complete_task(evidence) -> board_status.\n"
        "Never invent acceptance; evidence must cover acceptance strings. Chinese."
    )
    return create_agent(get_llm(), WB_TOOLS, system_prompt=system)


def format_agent_messages(messages: list[BaseMessage]) -> str:
    lines: list[str] = []
    for m in messages:
        if isinstance(m, HumanMessage):
            lines.append(f"USER: {m.content}")
        elif isinstance(m, AIMessage):
            if m.tool_calls:
                for tc in m.tool_calls:
                    lines.append(f"ACTION: {tc['name']}({tc.get('args') or {}})")
            if m.content:
                lines.append(f"ASSISTANT: {m.content}")
        elif isinstance(m, ToolMessage):
            content = m.content if len(str(m.content)) < 700 else str(m.content)[:700] + "..."
            lines.append(f"OBS[{m.name}]: {content}")
    return "\n".join(lines)


def scripted_prod_flow() -> dict[str, Any]:
    """
    不依赖模型的确定性生产路径（演示验收门）；有 key 时再跑 control agent。

    Returns:
        report: board + state 快照。
    """
    reset_prod_workbench()
    wb = _wb()
    t = wb.claim_next(session="script")
    assert t and t.id == "VAL"
    wb.touch("src/auth.py")
    # 故意缺证据
    bad = None
    try:
        wb.complete(evidence=["edited auth"])
    except RuntimeError as e:
        bad = str(e)
    wb.complete(evidence=["pytest green locally", "changed src/auth.py only"])
    t2 = wb.claim_next(session="script")
    return {
        "rejected": bad,
        "second_task": t2.id if t2 else None,
        "board": [asdict(x) for x in wb.read_board()],
        "handoff": wb.handoff_prompt(),
    }


print(f"LangChain minimal workbench ready | {MODEL}")


LangChain minimal workbench ready | deepseek:deepseek-v4-flash


## 4. 生产示例：认领 → 验收拒绝 → 通过 → 下一条

无 `DEEPSEEK_API_KEY` 则跳过 LLM control agent，仍跑脚本化工作台路径。


In [4]:
def demo_production_workbench() -> None:
    """生产：验收门 + 可选 DeepSeek 工具循环。"""
    rep = scripted_prod_flow()
    print("=== scripted workbench ===")
    print(json.dumps({k: rep[k] for k in ("rejected", "second_task", "board")}, ensure_ascii=False, indent=2))
    assert rep["rejected"] and "acceptance unmet" in rep["rejected"]
    assert rep["board"][0]["status"] == "done"
    assert rep["second_task"] == "DOC"
    print("acceptance + claim-next ok")

    if not os.getenv("DEEPSEEK_API_KEY"):
        print("SKIP llm agent: DEEPSEEK_API_KEY missing")
        print("PROD DEMO OK")
        return

    reset_prod_workbench()
    agent = build_control_agent()
    out = agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content=(
                        "reset_workbench，claim_task session=llm，"
                        "touch_file src/auth.py，"
                        "先用证据 ['edited'] 试 complete_task（应失败），"
                        "再用 ['pytest green', 'src/auth.py'] complete_task，"
                        "最后 board_status，用中文简述三文件各自角色。"
                    )
                )
            ]
        }
    )
    print("=== control agent ===")
    print(format_agent_messages(out.get("messages") or [])[:2000])
    board = _wb().read_board()
    assert board[0].status == "done"
    print("PROD DEMO OK")


demo_production_workbench()


=== scripted workbench ===
{
  "rejected": "acceptance unmet: ['pytest green', 'src/auth.py']",
  "second_task": "DOC",
  "board": [
    {
      "id": "VAL",
      "goal": "为登录接口补输入校验",
      "owner": "builder",
      "status": "done",
      "acceptance": [
        "pytest green",
        "src/auth.py"
      ]
    },
    {
      "id": "DOC",
      "goal": "更新 README 说明校验规则",
      "owner": "builder",
      "status": "in_progress",
      "acceptance": [
        "README"
      ]
    }
  ]
}
acceptance + claim-next ok
=== control agent ===
USER: reset_workbench，claim_task session=llm，touch_file src/auth.py，先用证据 ['edited'] 试 complete_task（应失败），再用 ['pytest green', 'src/auth.py'] complete_task，最后 board_status，用中文简述三文件各自角色。
ACTION: reset_workbench({})
OBS[reset_workbench]: {"root": "/var/folders/6g/9chj8ddx1033kwmzkbfd95l40000gn/T/min_wb_b88wt5z9"}
ACTION: claim_task({'session': 'llm'})
OBS[claim_task]: {"ok": true, "task": {"id": "VAL", "goal": "为登录接口补输入校验", "owner": "builder", "status": "in